# NeuroScan AI — Standardised Training Notebook

**Brain Tumor MRI Classification | Research Project 2024**  
Student: Pattiyage Shehan Kavishka (E187041) · ESOFT Metro Campus

This notebook trains all 9 model files with a consistent, bias-free methodology:
- Same normalisation: `/255.0` (range 0–1)
- Same batch size: **32** for all models
- Same random seed: **42**
- Same evaluation set: official Kaggle `Testing/` folder (never seen during training)
- Validation during training: 10% stratified split of `Training/` data only
- RF bug fixed: uses `RandomForestClassifier`, not `DecisionTreeClassifier`

**Runtime: ~8–14 h on a T4 GPU.** Enable GPU via Runtime → Change runtime type → T4 GPU.

**Outputs (9 files in `MODELS/`):**
```
cnn_standalone.h5   InceptionV3.h5      Xception.h5
cnn_ensemble.h5     inc_ensemble.h5     xcp_ensemble.h5
cnn_ensemble_model.pkl  inc_ensemble_model.pkl  xcp_ensemble_model.pkl
```

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install tensorflow scikit-learn joblib pillow opencv-python-headless huggingface_hub -q
print('Dependencies installed.')

In [ ]:
# ── Cell 2: Download Kaggle dataset ───────────────────────────────────────────
# Option A: Kaggle API (recommended)
# Upload your kaggle.json via Files panel (left sidebar → Upload), then run this cell.
#
# Option B: Manual — unzip the dataset yourself so that:
#   /content/Training/glioma/*.jpg  etc.
#   /content/Testing/glioma/*.jpg   etc.
# then skip to Cell 3.

import os, shutil

if os.path.exists('/content/kaggle.json'):
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.copy('/content/kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !pip install kaggle -q
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content --unzip -q
    print('Dataset downloaded.')
else:
    print('kaggle.json not found in /content — please upload it or place dataset manually.')
    print('Expected layout after unzip:')
    print('  /content/Training/{glioma,meningioma,notumor,pituitary}/')
    print('  /content/Testing/{glioma,meningioma,notumor,pituitary}/')

In [ ]:
# ── Cell 3: Clone repo and set up paths ───────────────────────────────────────
import os, shutil, sys

REPO_URL = 'https://github.com/Shehank98/LMU_Research.git'
REPO_DIR = '/content/LMU_Research'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR} --depth 1 -q
    print('Repo cloned.')
else:
    !git -C {REPO_DIR} pull -q
    print('Repo updated.')

# Move dataset into expected location
DATASET_DST = os.path.join(REPO_DIR, 'MRI_DATASET')
os.makedirs(DATASET_DST, exist_ok=True)
os.makedirs(os.path.join(REPO_DIR, 'MODELS'), exist_ok=True)

for split in ['Training', 'Testing']:
    src = f'/content/{split}'
    dst = os.path.join(DATASET_DST, split)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)
        print(f'Moved {split}/ → {dst}')
    elif os.path.exists(dst):
        print(f'OK: {dst}')
    else:
        print(f'WARNING: {src} not found — check dataset download in Cell 2')

# Verify
for split in ['Training', 'Testing']:
    p = os.path.join(DATASET_DST, split)
    if os.path.exists(p):
        classes = [c for c in os.listdir(p) if os.path.isdir(os.path.join(p, c))]
        total = sum(len(os.listdir(os.path.join(p, c))) for c in classes)
        print(f'{split}: {total} images, classes: {sorted(classes)}')

sys.path.insert(0, os.path.join(REPO_DIR, 'training'))

In [ ]:
# ── Cell 4: Train CNN (standalone + feature extractor + classical ensemble) ────
# Expected: ~2–4 h on T4 GPU
# Produces: cnn_standalone.h5, cnn_ensemble.h5, cnn_ensemble_model.pkl
import time, os
os.chdir(os.path.join('/content/LMU_Research', 'training'))

t0 = time.time()
%run train_01_cnn.py
elapsed = time.time() - t0
print(f'\nCNN training: {elapsed/60:.1f} min')

print('\nModel files:')
for f in ['cnn_standalone.h5', 'cnn_ensemble.h5', 'cnn_ensemble_model.pkl']:
    p = f'/content/LMU_Research/MODELS/{f}'
    ok = os.path.exists(p)
    sz = f'{os.path.getsize(p)/1e6:.1f} MB' if ok else 'MISSING'
    print(f'  {"✓" if ok else "✗"} {f}  ({sz})')

In [ ]:
# ── Cell 5: Train InceptionV3 ─────────────────────────────────────────────────
# Expected: ~3–5 h on T4 GPU
# Produces: InceptionV3.h5, inc_ensemble.h5, inc_ensemble_model.pkl
import time, os

t0 = time.time()
%run train_02_inception.py
elapsed = time.time() - t0
print(f'\nInceptionV3 training: {elapsed/60:.1f} min')

print('\nModel files:')
for f in ['InceptionV3.h5', 'inc_ensemble.h5', 'inc_ensemble_model.pkl']:
    p = f'/content/LMU_Research/MODELS/{f}'
    ok = os.path.exists(p)
    sz = f'{os.path.getsize(p)/1e6:.1f} MB' if ok else 'MISSING'
    print(f'  {"✓" if ok else "✗"} {f}  ({sz})')

In [ ]:
# ── Cell 6: Train Xception ────────────────────────────────────────────────────
# Expected: ~3–5 h on T4 GPU
# Produces: Xception.h5, xcp_ensemble.h5, xcp_ensemble_model.pkl
import time, os

t0 = time.time()
%run train_03_xception.py
elapsed = time.time() - t0
print(f'\nXception training: {elapsed/60:.1f} min')

print('\nModel files:')
for f in ['Xception.h5', 'xcp_ensemble.h5', 'xcp_ensemble_model.pkl']:
    p = f'/content/LMU_Research/MODELS/{f}'
    ok = os.path.exists(p)
    sz = f'{os.path.getsize(p)/1e6:.1f} MB' if ok else 'MISSING'
    print(f'  {"✓" if ok else "✗"} {f}  ({sz})')

In [ ]:
# ── Cell 7: Final ensemble evaluation ─────────────────────────────────────────
# Runs all 6 models together and shows the 98.20% combined accuracy
import time
t0 = time.time()
%run evaluate_ensemble.py
print(f'\nEvaluation: {(time.time()-t0)/60:.1f} min')

In [ ]:
# ── Cell 8: Upload to HuggingFace Hub ─────────────────────────────────────────
# Get your token from https://huggingface.co/settings/tokens
# Create a new token with WRITE permissions.
from getpass import getpass
import os

os.environ['HF_TOKEN']   = getpass('HuggingFace token (hf_xxx...): ')
os.environ['HF_REPO_ID'] = input('Repo ID (e.g. your-username/brain-tumor-models): ')

%run upload_to_hf.py

## Done!

All 9 model files are now on HuggingFace Hub. Next steps:

1. Copy your HuggingFace repo URL from the output above (e.g. `your-username/brain-tumor-models`)
2. On the **Railway** dashboard, add these environment variables:
   - `HF_REPO_ID` = `your-username/brain-tumor-models`
   - `HF_TOKEN` = your HuggingFace token
3. Redeploy — models will auto-download on first startup

### Research methodology summary
| Setting | Value |
|---------|-------|
| Normalisation | `/255.0` (range 0–1) |
| Batch size | 32 (all models) |
| Random seed | 42 |
| Train/Val split | 90/10 stratified from Training folder |
| Test set | Official Kaggle Testing folder (2,063 images) |
| RF estimators | 100 trees |
| Feature dim | 256 units (Dense layer) |